In [ ]:
import os
import warnings
from dotenv import load_dotenv
from llama_index.llms.google_genai import GoogleGenAI 
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, StorageContext, load_index_from_storage
from llama_index.readers.file import PyMuPDFReader #pdf reader
import logging
import nest_asyncio
nest_asyncio.apply()

os.environ["GOOGLE_API_KEY"] = "..." 


warnings.filterwarnings("ignore")

# Tắt tất cả các log INFO và WARNING từ các thư viện hệ thống
logging.basicConfig(level=logging.ERROR) 

# Tắt log cụ thể của các thư viện gây nhiễu
logging.getLogger("llama_index").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

os.environ["TQDM_DISABLE"] = "True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")#API key


Settings.llm = GoogleGenAI(model="models/gemini-3.1-flash-lite-preview", api_key=api_key)# AI model

Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2") #embedding model


Settings.chunk_size = 1024 #chunk size 
Settings.chunk_overlap = 100

PERSIST_DIR = "./storage" # data

def run_rag():
    
    file_extractor = {".pdf": PyMuPDFReader()} #pdf file extract 

    if os.path.exists(PERSIST_DIR):#load data embed neu ton tai roi
        print("1.data loading")
        storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
        index = load_index_from_storage(storage_context)
    else:
        print("2. indexing")#embed data roi luu 
        reader = SimpleDirectoryReader("./data", file_extractor=file_extractor)
        documents = reader.load_data()
        
        index = VectorStoreIndex.from_documents(documents, show_progress=True) 
        index.storage_context.persist(persist_dir=PERSIST_DIR)
        print(f"Save to {PERSIST_DIR}")


    query_engine = index.as_query_engine(similarity_top_k=10)#query config 
    
    question = """Câu 4. Khẳng định nào sau đây là sai?

A. Trong một đường tròn, góc nội tiếp chắn nửa đường tròn là góc vuông.

B. Trong một đường tròn, hai góc nội tiếp bằng nhau chắn hai cung bằng nhau.

C. Trong một đường tròn, hai góc nội tiếp cùng chắn một cung thì bằng nhau.

D. Trong một đường tròn, hai góc nội tiếp bằng nhau thì cùng chắn một cung.

"""
    print(f"\n--- Query: {question} ---")
    
    response = query_engine.query(question) # thuc hien query voi AI 
    print(f"\n GEMINI: {response}")

if __name__ == "__main__":

    if not os.path.exists(PERSIST_DIR):
        pass 
    else:
        print("Cảnh báo: Bạn nên xóa thư mục ./storage để hệ thống cập nhật dữ liệu PDF mới sạch hơn.")
    
    run_rag()

2026-05-06 22:43:28,558 - WARNING - Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6946.81it/s]


Cảnh báo: Bạn nên xóa thư mục ./storage để hệ thống cập nhật dữ liệu PDF mới sạch hơn.
1.data loading


2026-05-06 22:43:39,711 - INFO - AFC is enabled with max remote calls: 10.



--- Query: Câu 4. Khẳng định nào sau đây là sai?

A. Trong một đường tròn, góc nội tiếp chắn nửa đường tròn là góc vuông.

B. Trong một đường tròn, hai góc nội tiếp bằng nhau chắn hai cung bằng nhau.

C. Trong một đường tròn, hai góc nội tiếp cùng chắn một cung thì bằng nhau.

D. Trong một đường tròn, hai góc nội tiếp bằng nhau thì cùng chắn một cung.

 ---

 GEMINI: Answer: D


In [ ]:
import os
import logging
import warnings
import nest_asyncio
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, StorageContext, load_index_from_storage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.readers.llama_parse import LlamaParse
from llama_index.llms.google_genai import GoogleGenAI

# 1. Cấu hình hệ thống
nest_asyncio.apply()
logging.basicConfig(level=logging.ERROR)
for logger_name in ["llama_index", "httpx", "sentence_transformers", "huggingface_hub"]:
    logging.getLogger(logger_name).setLevel(logging.ERROR)

os.environ["TQDM_DISABLE"] = "True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# 2. CẤU HÌNH API KEY (Điền Key của bạn vào đây)
LLAMA_CLOUD_KEY = "..."

os.environ["GOOGLE_API_KEY"] = "..." # Thay bằng Key Google AI Studio của bạn
os.environ["LLAMA_CLOUD_API_KEY"] = LLAMA_CLOUD_KEY

# 3. Cấu hình Model & Embedding
Settings.llm = GoogleGenAI(model="models/gemini-3.1-flash-lite-preview") 
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    embed_batch_size=42 # Tối ưu cho GPU của M1
)
Settings.chunk_size = 1024
Settings.chunk_overlap = 100


# 3. Cấu hình LlamaParse chuyên cho tài liệu scan
parser = LlamaParse(
    result_type="markdown",  # Ép kiểu về Markdown để AI hiểu cấu trúc bảng/tiêu đề
    num_workers=8,
    gpt4o_mode=True,
    verbose=True,
    language="en"            # Cuốn sách của bạn là tiếng Anh
)

PERSIST_DIR = "./storage"
DATA_DIR = "./data"

# 5. Kiểm tra và khởi tạo Index
if not os.path.exists(PERSIST_DIR):
    print(f"🚀 Đang quét dữ liệu từ: {DATA_DIR}...")
    
    # Cấu hình Parser
    parser = LlamaParse(result_type="markdown", num_workers=4) # Giảm worker xuống 4 để ổn định hơn
    file_extractor = {".pdf": parser}
    
    # Nạp dữ liệu
    reader = SimpleDirectoryReader(input_dir=DATA_DIR, file_extractor=file_extractor)
    
    # SỬA TẠI ĐÂY: Sử dụng load_data() không có tham số để tránh xung đột async
    documents = reader.load_data()
    
    if not documents or len(documents) == 0:
        print("❌ Lỗi: Không thể trích xuất nội dung từ file. Hãy kiểm tra lại API Key.")
    else:
        print(f"✅ Đã nạp thành công {len(documents)} trang/tài liệu.")
        index = VectorStoreIndex.from_documents(documents)
        index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    # ... (giữ nguyên phần load từ storage)
    print("📂 Đang tải dữ liệu từ bộ nhớ local...")
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# 6. Truy vấn
query_engine = index.as_query_engine(similarity_top_k=5)

/Users/nguyenhoangminh/Documents/DUT_AI/.venv_ai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6642.00it/s]


📂 Đang tải dữ liệu từ bộ nhớ local...

🤖 ĐÁP ÁN: Dựa trên nội dung được cung cấp, tư tưởng Hồ Chí Minh được hình thành từ sự kết hợp giữa chủ nghĩa yêu nước và chủ nghĩa Mác - Lênin trong một bối cảnh cụ thể:

**1. Về cơ sở lý luận và nền tảng tư tưởng:**
*   **Chủ nghĩa Mác - Lênin:** Đóng vai trò quyết định trong việc định hình nội dung và bản chất cách mạng của tư tưởng Hồ Chí Minh. Chủ nghĩa Mác - Lênin cung cấp thế giới quan, phương pháp luận khoa học, giúp nhận diện bản chất kẻ thù, quy luật vận động của lịch sử và các phương pháp cách mạng. Hồ Chí Minh đã tiếp thu chủ nghĩa Mác - Lênin bằng cách nắm lấy tinh thần và bản chất, từ đó vận dụng sáng tạo phù hợp với điều kiện, hoàn cảnh cụ thể của cách mạng Việt Nam.
*   **Chủ nghĩa yêu nước:** Là cội nguồn sâu xa hình thành nên tư tưởng Hồ Chí Minh. Truyền thống văn hóa dân tộc và khát vọng giải phóng dân tộc chính là động lực thôi thúc Người ra đi tìm đường cứu nước, tạo bản lĩnh và trí tuệ để tiếp thu tinh hoa nhân loại.

**2. Về 

In [ ]:
import os
import shutil
import logging
import warnings
import nest_asyncio
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, StorageContext, load_index_from_storage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.readers.llama_parse import LlamaParse
from llama_index.llms.google_genai import GoogleGenAI

# 1. Cấu hình hệ thống
nest_asyncio.apply()
logging.basicConfig(level=logging.ERROR)
for logger_name in ["llama_index", "httpx", "sentence_transformers", "huggingface_hub"]:
    logging.getLogger(logger_name).setLevel(logging.ERROR)

os.environ["TQDM_DISABLE"] = "True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# 2. CẤU HÌNH API KEY (Điền Key của bạn vào đây)
LLAMA_CLOUD_KEY = "..."

os.environ["GOOGLE_API_KEY"] = "..." # Thay bằng Key Google AI Studio của bạn
os.environ["LLAMA_CLOUD_API_KEY"] = LLAMA_CLOUD_KEY

# 3. Cấu hình Model & Embedding
Settings.llm = GoogleGenAI(model="models/gemini-3.1-flash-lite-preview") 
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    embed_batch_size=42 # Tối ưu cho GPU của M1
)
Settings.chunk_size = 1024
Settings.chunk_overlap = 100

# --- Cấu hình đường dẫn ---
PERSIST_DIR = "./storage"
DATA_DIR = "./data"

# Bước 0: Reset storage cũ để nạp lại từ đầu (Chỉ chạy khi thêm file mới)
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)
    print("🧹 Đã dọn dẹp bộ nhớ cũ.")

# Bước 1: Khởi tạo Parser chuyên dụng cho PDF
parser = LlamaParse(
    result_type="markdown", 
    num_workers=4, 
    verbose=True,
    language="en"
)

# Bước 2: Nạp dữ liệu thủ công để kiểm soát lỗi
all_docs = []

print(f"🚀 Bắt đầu quét thư mục: {DATA_DIR}")

# A. Nạp các file Word (.docx) - Xử lý Offline cực nhanh
word_reader = SimpleDirectoryReader(input_dir=DATA_DIR, required_exts=[".docx"])
word_docs = word_reader.load_data()
all_docs.extend(word_docs)
print(f"✅ Đã nạp file Word: {[d.metadata.get('file_name') for d in word_docs][:1]}...")

# B. Nạp các file PDF (.pdf) - Xử lý qua LlamaParse OCR
pdf_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith('.pdf')]
for pdf_name in pdf_files:
    pdf_path = os.path.join(DATA_DIR, pdf_name)
    print(f"📡 Đang gửi PDF lên Cloud để OCR: {pdf_name}...")
    try:
        # Gọi trực tiếp parser
        pdf_docs = parser.load_data(pdf_path)
        
        # ÉP METADATA: Gắn tên file vào từng trang để AI nhớ nguồn
        for doc in pdf_docs:
            doc.metadata["file_name"] = pdf_name
            # LlamaParse thường trả về số trang trong doc.metadata['page_number']
            # Nếu không có, ta giữ nguyên N/A hoặc dùng số thứ tự
            
        all_docs.extend(pdf_docs)
        print(f"✅ Đã nạp xong PDF: {pdf_name}")
    except Exception as e:
        print(f"❌ Lỗi khi đọc PDF {pdf_name}: {e}")
# Bước 3: Tạo Index chung cho tất cả tài liệu
if all_docs:
    index = VectorStoreIndex.from_documents(all_docs)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    print(f"✨ HOÀN THÀNH! Đã lưu {len(all_docs)} mảnh dữ liệu vào storage.")
    query_engine = index.as_query_engine(similarity_top_k=5)
else:
    print("⚠️ Cảnh báo: Không có dữ liệu nào được nạp!")

🧹 Đã dọn dẹp bộ nhớ cũ.
🚀 Bắt đầu quét thư mục: ./data
✅ Đã nạp file Word: ['Đề cương chi tiết bài giảng TT HCM 2026.docx']...
📡 Đang gửi PDF lên Cloud để OCR: Computer Networking - 3rd edition.pdf...
Started parsing the file under job_id 591e23b0-2d0f-4acb-b123-4c77db9f9dbb
✅ Đã nạp xong PDF: Computer Networking - 3rd edition.pdf
📡 Đang gửi PDF lên Cloud để OCR: Thay_Nguyen Tan Khoi.pdf...
Error while parsing the file './data/Thay_Nguyen Tan Khoi.pdf': Event loop is closed
✅ Đã nạp xong PDF: Thay_Nguyen Tan Khoi.pdf
✨ HOÀN THÀNH! Đã lưu 849 mảnh dữ liệu vào storage.


In [10]:
def ask_with_source(question):
    response = query_engine.query(question)
    
    print("\n" + "="*50)
    print(f"🤖 ĐÁP ÁN: {response}")
    print("="*50)
    
    print("\n📍 CHI TIẾT NGUỒN DỮ LIỆU:")
    
    seen_sources = set()
    for i, node in enumerate(response.source_nodes):
        meta = node.node.metadata
        
        # Thử lấy tên file từ nhiều key khác nhau
        f_name = meta.get('file_name') or meta.get('file_path') or "Tài liệu không tên"
        f_name = os.path.basename(f_name) # Chỉ lấy tên file, bỏ đường dẫn dài
        
        # Thử lấy số trang
        page = meta.get('page_number') or meta.get('page') or meta.get('page_label') or "N/A"
        
        source_id = f"{f_name}_{page}"
        if source_id not in seen_sources:
            print(f"[{i+1}] 📄 File: {f_name} | 📍 Trang: {page} | ✨ Khớp: {node.score:.2%}")
            # In thêm một đoạn ngắn để kiểm chứng
            content = node.node.get_content().replace('\n', ' ')[:150]
            print(f"    📝 Nội dung: \"{content}...\"")
            seen_sources.add(source_id)

In [11]:
# 6. Chạy thử nghiệm
# Ví dụ: Bạn có file 'ly_thuyet.pdf' và 'baitap.docx' trong thư mục data
ask_with_source("1. Phân tích bối cảnh lịch sử trong nước và quốc tế dẫn đến ra đời Tư tưởng Hồ Chí Minh?")
ask_with_source("2. Thiết bị nào cung cấp kết nối giữa các máy tính trong cùng mạng LAN ?")


🤖 ĐÁP ÁN: Dựa trên các nội dung được cung cấp, quá trình hình thành tư tưởng Hồ Chí Minh gắn liền với các yếu tố cơ bản sau:

**1. Về cội nguồn văn hóa và truyền thống:**
*   **Văn hóa dân tộc:** Tinh hoa văn hóa dân tộc là cội nguồn sâu xa, tạo nên bản lĩnh và tầm trí tuệ để Hồ Chí Minh tiếp thu tinh hoa văn hóa nhân loại. Khát vọng giải phóng dân tộc từ truyền thống yêu nước đã thôi thúc Người ra đi tìm đường cứu nước.
*   **Văn hóa nhân loại:** Người tiếp thu chọn lọc tinh hoa văn hóa phương Đông (đặc biệt là các giá trị tích cực về tu dưỡng đạo đức và quản lý xã hội từ Nho giáo sau khi đã phê phán các mặt hạn chế) và tinh hoa văn hóa phương Tây.

**2. Vai trò của chủ nghĩa Mác - Lênin:**
Đây là cơ sở lý luận đóng vai trò quyết định đối với nội dung và bản chất cách mạng của tư tưởng Hồ Chí Minh. Chủ nghĩa Mác - Lênin giúp Người:
*   Có thế giới quan và phương pháp luận để nhận diện bản chất kẻ thù, đề ra đường lối chiến lược, sách lược và nhận thức quy luật vận động của lịch sử.
*

In [ ]:
import os
import logging
import warnings
import nest_asyncio
from llama_parse import LlamaParse
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, load_index_from_storage, Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

nest_asyncio.apply()
logging.basicConfig(level=logging.ERROR)
for logger_name in ["llama_index", "httpx", "sentence_transformers", "huggingface_hub"]:
    logging.getLogger(logger_name).setLevel(logging.ERROR)

os.environ["TQDM_DISABLE"] = "True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# 1. Khai báo API Keys
os.environ["GOOGLE_API_KEY"] = "..." # Thay bằng Key Google AI Studio của bạn
os.environ["LLAMA_CLOUD_API_KEY"] = "..." # Thay bằng Key LlamaIndex của bạn

# 2. Cấu hình Model
Settings.llm = GoogleGenAI(model="models/gemini-3.1-flash-lite-preview")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 3. Cấu hình LlamaParse chuyên cho tài liệu scan
parser = LlamaParse(
    result_type="markdown",
    num_workers=8,
    verbose=True,
    language="en",
    # THÊM DÒNG NÀY: Ép buộc sử dụng OCR cấp độ cao cho file scan
    gpt4o_mode=True, 
    parsing_instruction="This is a scanned PDF of a computer networking textbook. Please extract all text, headings, and tables accurately."
)

PERSIST_DIR = "./storage"
DATA_FILE = "./data/Computer Networking - 3rd edition.pdf"

# 4. Xử lý dữ liệu
if not os.path.exists(PERSIST_DIR):
    print("🚀 LlamaParse đang phân tích cú pháp file scan (vui lòng đợi)...")
    
    # Chỉ định parser cho file PDF
    file_extractor = {".pdf": parser}
    reader = SimpleDirectoryReader(input_files=[DATA_FILE], file_extractor=file_extractor)
    documents = reader.load_data()
    
    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    print("✅ Đã lưu nội dung scan vào bộ nhớ!")
else:
    print("📂 Đang tải dữ liệu đã phân tích từ trước...")
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)

# 5. Truy vấn và Hiển thị
query_engine = index.as_query_engine(similarity_top_k=5) #Truy vấn nâng cao (Lấy 5 nguồn để trích dẫn đầy đủ hơn)
question = """Thiết bị nào sau đây cung cấp kết nối giữa các máy tính trong cùng mạng LAN.
                a.NIC
                b.router
                c.switch
                d.amplifier"""
print(f"\nCâu hỏi: {question}")
response = query_engine.query(question)

# 7. HIỂN THỊ KẾT QUẢ KÈM NGUỒN VÀ TRANG
print("\n" + "="*70)
print(f"Đáp án:\n{response}")
print("="*70)

print("\n" + " DANH SÁCH NGUỒN TRÍCH DẪN ".center(70, "-"))

# Dùng dictionary để nhóm nội dung theo số trang (tránh lặp trang)
source_details = {}

for node in response.source_nodes:
    metadata = node.node.metadata
    # LlamaParse thường dùng 'page_number', một số reader khác dùng 'page'
    page = metadata.get('page_number') or metadata.get('page') or "N/A"
    file_name = metadata.get('file_name', 'Giao trinh Mang may tinh')
    
    if page not in source_details:
        source_details[page] = {
            "file": file_name,
            "score": node.score,
            "text": node.node.get_content().strip()[:250] # Lấy 250 ký tự đầu của trang
        }

# In thông tin nguồn
for page, info in sorted(source_details.items(), key=lambda x: str(x[0])):
    print(f"\n📍 TRANG: {page} | Độ khớp: {info['score']:.2%}")
    print(f"📄 File: {info['file']}")
    print(f"📝 Trích đoạn: \"{info['text']}...\"")
    print("-" * 40)

print("\n" + " KẾT THÚC TRÍCH DẪN ".center(70, "-"))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6548.87it/s]


📂 Đang tải dữ liệu đã phân tích từ trước...

Câu hỏi: Thiết bị nào sau đây cung cấp kết nối giữa các máy tính trong cùng mạng LAN.
                a.NIC
                b.router
                c.switch
                d.amplifier

Đáp án:
c.switch

--------------------- DANH SÁCH NGUỒN TRÍCH DẪN ----------------------

📍 TRANG: N/A | Độ khớp: 67.28%
📄 File: Giao trinh Mang may tinh
📝 Trích đoạn: "# CHAPTER 1 COMPUTER NETWORKS AND THE INTERNET

than the more traditional circuit-switching technology used in the telephone networks. For this reason, we now give a brief overview of circuit switching.

Figure 1.5 A simple circuit-switched network c..."
----------------------------------------

------------------------- KẾT THÚC TRÍCH DẪN -------------------------


In [6]:
import os
import shutil

# Bước 0: Dọn dẹp storage cũ để đảm bảo không load lại dữ liệu rỗng
if os.path.exists("./storage"):
    shutil.rmtree("./storage")

# Bước 1: Khởi tạo lại Parser với cấu hình OCR mạnh nhất
# LlamaParse sẽ tự động nhận diện file scan và chuyển sang OCR
parser = LlamaParse(
    result_type="markdown",  # BẮT BUỘC để có nội dung text
    verbose=True,
    language="en",
    num_workers=4
)

# Bước 2: Sử dụng phương thức chuẩn load_data
print("🚀 Đang gửi file lên LlamaCloud để phân tích OCR...")
# load_data trả về một danh sách các đối tượng Document đã có sẵn Text
documents = parser.load_data("./data/Computer Networking - 3rd edition.pdf")

# Bước 3: Kiểm tra dữ liệu ngay lập tức
if len(documents) > 0 and documents[0].get_content().strip():
    print(f"✅ Thành công! Đã quét được {len(documents)} trang.")
    print(f"📝 Trích đoạn trang 1: {documents[0].get_content()[:200]}...")
    
    # Tiếp tục tạo Index
    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir="./storage")
    query_engine = index.as_query_engine()
    
    # Chạy thử câu hỏi
    response = query_engine.query("Thiết bị nào kết nối máy tính trong cùng mạng LAN?")
    print(f"\n🤖 Đáp án: {response}")
else:
    print("❌ Vẫn không lấy được văn bản. Hãy kiểm tra lại file PDF hoặc API Key.")

🚀 Đang gửi file lên LlamaCloud để phân tích OCR...
Started parsing the file under job_id af05d9cd-110a-482c-8fd7-6113f34232c3
...........✅ Thành công! Đã quét được 848 trang.
📝 Trích đoạn trang 1: INTERNATIONAL Edition
# A TOP-DOWN APPROACH

# FEATURING THE INTERNET

# networking

# third edition

James F. Kurose

Keith W. Ross
...

🤖 Đáp án: Các thiết bị được sử dụng để kết nối các máy tính trong mạng LAN bao gồm hub (bộ tập trung) và switch (thiết bị chuyển mạch lớp liên kết dữ liệu). Ngoài ra, các mạng LAN cũng có thể sử dụng công nghệ token-passing (như token ring) nơi các nút được kết nối trực tiếp với nhau theo cấu trúc vòng.


In [7]:
def ask_question(question):
    response = query_engine.query(question)
    
    print(f"\n🤖 AI trả lời: {response}")
    print("\n" + "-"*30)
    print("📚 Nguồn trích dẫn:")
    
    # Lấy danh sách trang không trùng lặp
    pages = sorted(list(set([node.node.metadata.get('page_number') for node in response.source_nodes])))
    
    for p in pages:
        if p: print(f"📍 Tham khảo tại trang: {p}")
    
    # In một đoạn ngắn của nguồn khớp nhất để kiểm chứng
    if response.source_nodes:
        top_node = response.source_nodes[0]
        print(f"\n📖 Nội dung khớp nhất (Trang {top_node.node.metadata.get('page_number')}):")
        print(f"'{top_node.node.get_content()[:200]}...'")

# Sử dụng
ask_question("Trình bày về mô hình OSI và chức năng của tầng Link Layer")


🤖 AI trả lời: Tầng liên kết dữ liệu (link layer) có nhiệm vụ chính là vận chuyển các khung (frame) hoàn chỉnh từ một phần tử mạng này sang một phần tử mạng liền kề. Khác với dịch vụ đầu-cuối (end-to-end), các dịch vụ của tầng này được cung cấp trên cơ sở từng nút (node-to-adjacent-node).

Các cơ chế của tầng liên kết dữ liệu, chẳng hạn như giao thức truy cập ngẫu nhiên hoặc các quy trình đảm bảo phân phối dữ liệu tin cậy (như đánh số thứ tự, bộ định thời và xác nhận), thường được triển khai hoàn toàn trong các bộ điều hợp (adapter), hay còn gọi là card mạng (NIC).

Về cấu trúc phân tầng, các thiết bị mạng có sự khác biệt trong việc triển khai các tầng của giao thức:
- Các máy chủ (hosts) triển khai đầy đủ năm tầng của giao thức.
- Các bộ định tuyến (routers) triển khai các tầng từ 1 đến 3, cho phép chúng thực hiện các giao thức mạng như IP.
- Các thiết bị chuyển mạch liên kết (link-layer switches) chỉ triển khai tầng 1 và tầng 2. Do đó, chúng không nhận diện địa chỉ IP mà chỉ nhận diệ